In [1]:
# Instalação do Apache Spark para Python.
# Fixamos a versão para facilitar a reprodução do trabalho.
%pip install -q "pyspark==4.0.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# Importação das ferramentas utilizadas na configuração.
import os
import sys
import shutil
import subprocess
from pathlib import Path

# Verifica a presença do Java e configura seu caminho.
caminho_java = shutil.which("java")

if caminho_java is None:
    raise RuntimeError(
        "Java não encontrado. Envie esta mensagem para ajustarmos o ambiente."
    )

os.environ["JAVA_HOME"] = str(
    Path(caminho_java).resolve().parent.parent
)

# Garante que o Spark utilize o Python deste notebook.
os.environ["PYSPARK_PYTHON"] = sys.executable

verificacao_java = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True,
    check=True
)

print("Python:", sys.version.split()[0])
print("Java:")
print(verificacao_java.stderr or verificacao_java.stdout)

# Inicialização do Spark.
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Sistematizacao_Ciencia_de_Dados_II")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Versão do Spark:", spark.version)

# Dados fictícios usados somente para testar o ambiente.
teste = spark.createDataFrame(
    [
        ("A", 10.0),
        ("B", 20.0),
        ("A", 30.0)
    ],
    ["grupo", "valor"]
)

# Registro de uma view e execução de uma consulta Spark SQL.
teste.createOrReplaceTempView("teste_spark")

spark.sql("""
    SELECT
        COUNT(*) AS registros,
        SUM(valor) AS total
    FROM teste_spark
""").show()

print("TESTE CONCLUÍDO: Spark e Spark SQL funcionando.")

Python: 3.13.15
Java:
openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)

Versão do Spark: 4.0.1
+---------+-----+
|registros|total|
+---------+-----+
|        3| 60.0|
+---------+-----+

TESTE CONCLUÍDO: Spark e Spark SQL funcionando.


# Sistematização — Ciência de Dados II

## Qualidade do ar: estimativa de PM2.5 e identificação de perfis de poluição com Apache Spark

### 1. Seleção do dataset e definição do problema

Este projeto aplica o processo de KDD a um estudo de caso de consultoria
ambiental, utilizando Apache Spark para preparação, exploração e modelagem
dos dados.

O dataset escolhido é o Beijing Multi-Site Air Quality, disponibilizado
pelo UCI Machine Learning Repository. A escolha considera o volume
superior a 100 mil registros, a documentação da fonte e a possibilidade
de desenvolver análises supervisionadas e não supervisionadas.

Fonte original:
https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data

Referência:
Chen, S. (2017). Beijing Multi-Site Air Quality [Dataset].
UCI Machine Learning Repository.
https://doi.org/10.24432/C5RK5G

### Objetivos

Investigar padrões temporais e espaciais nas medições, comparar modelos
para estimar a concentração de PM2.5 e identificar grupos de observações
com perfis semelhantes de poluição.

### Delimitação

A modelagem supervisionada será uma estimativa contemporânea:
utilizará outras informações disponíveis no horário da observação,
sem utilizar o próprio PM2.5 como variável de entrada.

O estudo é histórico e exploratório. Seus resultados não demonstram
relações de causa e efeito nem constituem um sistema operacional validado.

In [3]:
# ETAPA 1 — Download e organização dos arquivos originais

from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import shutil
import zipfile

URL_DATASET = (
    "https://archive.ics.uci.edu/static/public/501/"
    "beijing%2Bmulti%2Bsite%2Bair%2Bquality%2Bdata.zip"
)

PASTA_DADOS = Path("/content/dados_qualidade_ar")
PASTA_ORIGINAIS = PASTA_DADOS / "originais"
ARQUIVO_ZIP = PASTA_DADOS / "fonte_uci.zip"

PASTA_DADOS.mkdir(parents=True, exist_ok=True)
PASTA_ORIGINAIS.mkdir(parents=True, exist_ok=True)

# Reutiliza o download caso o ZIP já esteja disponível e seja válido.
if not ARQUIVO_ZIP.exists() or not zipfile.is_zipfile(ARQUIVO_ZIP):
    print("Baixando o dataset da fonte oficial...")

    requisicao = Request(
        URL_DATASET,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    arquivo_temporario = ARQUIVO_ZIP.with_suffix(".tmp")

    with urlopen(requisicao, timeout=120) as resposta:
        with arquivo_temporario.open("wb") as destino:
            shutil.copyfileobj(resposta, destino)

    if not zipfile.is_zipfile(arquivo_temporario):
        raise RuntimeError(
            "O download não retornou um ZIP válido. Envie esta mensagem."
        )

    arquivo_temporario.replace(ARQUIVO_ZIP)

else:
    print("ZIP já disponível. Reutilizando o arquivo.")


def extrair_zip_seguro(arquivo: Path, destino: Path) -> None:
    """Extrai um ZIP verificando se os caminhos ficam dentro do destino."""
    destino = destino.resolve()
    destino.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(arquivo) as pacote:
        for item in pacote.infolist():
            caminho_final = (destino / item.filename).resolve()

            if not caminho_final.is_relative_to(destino):
                raise RuntimeError(
                    f"Caminho inesperado dentro do ZIP: {item.filename}"
                )

        pacote.extractall(destino)


extrair_zip_seguro(ARQUIVO_ZIP, PASTA_ORIGINAIS)

# A distribuição pode conter outro ZIP dentro do arquivo principal.
for zip_interno in list(PASTA_ORIGINAIS.rglob("*.zip")):
    extrair_zip_seguro(zip_interno, zip_interno.with_suffix(""))

# Seleciona apenas os arquivos originais das estações.
# Outros CSVs eventualmente presentes no pacote não entram na análise.
arquivos_csv = sorted(PASTA_ORIGINAIS.rglob("PRSA_Data_*.csv"))

if len(arquivos_csv) != 12:
    raise RuntimeError(
        f"Foram encontrados {len(arquivos_csv)} arquivos de estações, "
        "mas eram esperados 12. Envie esta saída antes de continuar."
    )

# Identificador do arquivo baixado para apoiar a rastreabilidade.
sha256 = hashlib.sha256(ARQUIVO_ZIP.read_bytes()).hexdigest()

print(f"\nArquivos de estações encontrados: {len(arquivos_csv)}")
print(f"Tamanho do ZIP: {ARQUIVO_ZIP.stat().st_size / 1024**2:.2f} MiB")
print(f"SHA-256 do ZIP: {sha256}")

print("\nArquivos selecionados:")
for arquivo in arquivos_csv:
    print("-", arquivo.name)

print("\nDOWNLOAD E EXTRAÇÃO CONCLUÍDOS.")

Baixando o dataset da fonte oficial...

Arquivos de estações encontrados: 12
Tamanho do ZIP: 7.81 MiB
SHA-256 do ZIP: b04da438b2f331ac0ffd45aebdfec0d20d2367feb5f6948c4b1f7ce1191e33c4

Arquivos selecionados:
- PRSA_Data_Aotizhongxin_20130301-20170228.csv
- PRSA_Data_Changping_20130301-20170228.csv
- PRSA_Data_Dingling_20130301-20170228.csv
- PRSA_Data_Dongsi_20130301-20170228.csv
- PRSA_Data_Guanyuan_20130301-20170228.csv
- PRSA_Data_Gucheng_20130301-20170228.csv
- PRSA_Data_Huairou_20130301-20170228.csv
- PRSA_Data_Nongzhanguan_20130301-20170228.csv
- PRSA_Data_Shunyi_20130301-20170228.csv
- PRSA_Data_Tiantan_20130301-20170228.csv
- PRSA_Data_Wanliu_20130301-20170228.csv
- PRSA_Data_Wanshouxigong_20130301-20170228.csv

DOWNLOAD E EXTRAÇÃO CONCLUÍDOS.


In [4]:
# ETAPA 2 — Ingestão dos arquivos com Spark DataFrames

from pyspark import StorageLevel
from pyspark.sql import functions as F

# Esquema correspondente à ordem das colunas nos arquivos originais.
esquema_original = """
    No INT,
    year INT,
    month INT,
    day INT,
    hour INT,
    `PM2.5` DOUBLE,
    PM10 DOUBLE,
    SO2 DOUBLE,
    NO2 DOUBLE,
    CO DOUBLE,
    O3 DOUBLE,
    TEMP DOUBLE,
    PRES DOUBLE,
    DEWP DOUBLE,
    RAIN DOUBLE,
    wd STRING,
    WSPM DOUBLE,
    station STRING
"""

df_original = (
    spark.read
    .schema(esquema_original)
    .option("header", True)
    .option("nullValue", "NA")
    .option("mode", "FAILFAST")
    .option("enforceSchema", False)
    .csv([str(arquivo) for arquivo in arquivos_csv])
)

# Padronização dos nomes para facilitar consultas e documentação.
df_bruto = df_original.toDF(
    "numero",
    "ano",
    "mes",
    "dia",
    "hora",
    "pm25",
    "pm10",
    "so2",
    "no2",
    "co",
    "o3",
    "temperatura",
    "pressao",
    "ponto_orvalho",
    "precipitacao",
    "direcao_vento",
    "velocidade_vento",
    "estacao"
)

# Mantém o DataFrame disponível para as próximas verificações.
df_bruto = df_bruto.persist(StorageLevel.MEMORY_AND_DISK)

total_registros = df_bruto.count()
total_estacoes = df_bruto.select("estacao").distinct().count()

if total_registros == 0:
    raise RuntimeError("Nenhum registro foi carregado.")

print("Registros carregados:", f"{total_registros:,}".replace(",", "."))
print("Quantidade de colunas:", len(df_bruto.columns))
print("Estações encontradas:", total_estacoes)

print("\nESTRUTURA DO DATAFRAME:")
df_bruto.printSchema()

print("\nEXEMPLOS DE REGISTROS:")
df_bruto.select(
    "estacao", "ano", "mes", "dia", "hora",
    "pm25", "pm10", "temperatura"
).show(5, truncate=False)

print("\nINGESTÃO COM PYSPARK CONCLUÍDA.")

Registros carregados: 420.768
Quantidade de colunas: 18
Estações encontradas: 12

ESTRUTURA DO DATAFRAME:
root
 |-- numero: integer (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- hora: integer (nullable = true)
 |-- pm25: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- so2: double (nullable = true)
 |-- no2: double (nullable = true)
 |-- co: double (nullable = true)
 |-- o3: double (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- pressao: double (nullable = true)
 |-- ponto_orvalho: double (nullable = true)
 |-- precipitacao: double (nullable = true)
 |-- direcao_vento: string (nullable = true)
 |-- velocidade_vento: double (nullable = true)
 |-- estacao: string (nullable = true)


EXEMPLOS DE REGISTROS:
+-------------+----+---+---+----+----+----+-----------+
|estacao      |ano |mes|dia|hora|pm25|pm10|temperatura|
+-------------+----+---+---+----+----+----+-----------+

In [5]:
# ETAPA 2 — Diagnóstico inicial de valores ausentes

contagem_ausentes = (
    df_bruto
    .agg(*[
        F.sum(
            F.col(coluna).isNull().cast("long")
        ).alias(coluna)
        for coluna in df_bruto.columns
    ])
    .first()
    .asDict()
)

linhas_diagnostico = []

for coluna in df_bruto.columns:
    quantidade = int(contagem_ausentes[coluna] or 0)
    percentual = round(100.0 * quantidade / total_registros, 2)

    linhas_diagnostico.append(
        (coluna, quantidade, percentual)
    )

diagnostico_ausentes = spark.createDataFrame(
    linhas_diagnostico,
    schema="coluna STRING, ausentes LONG, percentual DOUBLE"
)

print("VALORES AUSENTES POR COLUNA:")

diagnostico_ausentes.orderBy(
    F.desc("ausentes"),
    F.asc("coluna")
).show(len(df_bruto.columns), truncate=False)

VALORES AUSENTES POR COLUNA:
+----------------+--------+----------+
|coluna          |ausentes|percentual|
+----------------+--------+----------+
|co              |20701   |4.92      |
|o3              |13277   |3.16      |
|no2             |12116   |2.88      |
|so2             |9021    |2.14      |
|pm25            |8739    |2.08      |
|pm10            |6449    |1.53      |
|direcao_vento   |1822    |0.43      |
|ponto_orvalho   |403     |0.1       |
|temperatura     |398     |0.09      |
|pressao         |393     |0.09      |
|precipitacao    |390     |0.09      |
|velocidade_vento|318     |0.08      |
|ano             |0       |0.0       |
|dia             |0       |0.0       |
|estacao         |0       |0.0       |
|hora            |0       |0.0       |
|mes             |0       |0.0       |
|numero          |0       |0.0       |
+----------------+--------+----------+

